In [0]:
env = dbutils.widgets.get("environment") if dbutils.widgets.getAll().get("environment") else "dev"
targetTable = f"saleslake_{env}.bronze_{env}.rawSales"
srcPath = f"/Volumes/saleslake_{env}/bronze_{env}/vol_saleslake_src_files_{env}/daily_sales/"

spark.sql(f"""
    COPY INTO {targetTable}
    FROM (
        SELECT
            sale_id,
            product_id,
            store_id,
            customer_id,
            region_id,
            quantity,
            unit_price,
            gross_amount,
            discount_code,
            discount_amount,
            net_amount,
            cost_amount,
            sale_date,
            payment_method,
            channel,
            current_timestamp() AS ingest_ts
        FROM '{srcPath}'
    )
    FILEFORMAT = CSV
    FORMAT_OPTIONS (
        'header' = 'true'
    )
    COPY_OPTIONS (
        'mergeSchema' = 'true'
    )
    """)

print("Sales Bronze Load Successful")

spark.sql(f"""
    SELECT COUNT(*) AS total_records
    FROM {targetTable}
    """).show()